# SelectOmics · GBM CNV

TCGA glioblastoma, 244 samples, 11,205 CNV features,
5 subtypes. Copy-number variation, around 11 000 segments. Values are discrete and highly correlated along the genome, so neighbouring segments carry near-identical signal.

**What this notebook shows.** A full run with every evaluation switched on, and
the panel the pipeline *recommends* rather than whichever step happened to run
last. The two differ when a later step prunes past the point where it helps.

| Section | Shows |
|---|---|
| 3 | The configuration, and what `suggest()` proposed |
| 5 | Each step against the Step 0 reference |
| 6 | The three validation protocols, and the recommendation |
| 7 | The held-out test, scored on the recommended panel |

| 8 | Feature provenance |
| 9 | Every figure and file the run wrote |

**Runtime.** About **69 minutes** at the settings in Section 3, on a
24-core desktop. Almost all of it is Step 3: it refits the model once per
feature it is asked to eliminate, once per consensus model, once per inner
fold, and once per probe of its retention search. A layer where Step 2 already
cuts below Step 3's gate of 30 features finishes far quicker, which is why the
11,205-feature layers here are not the slowest ones. Switching `algorithm` to
`"RF"` cuts it further: the same GBM miRNA layer runs in about 5 minutes under
RF in `GBM_Quickstart.ipynb`, against 247 under XGB, because boosting cannot
parallelise across trees the way a forest can.

## 1 · Setup

In [ ]:
# Adds the package to sys.path for this session and registers it with pip for
# later ones. No kernel restart needed.
import subprocess
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

_pkg_root = str(Path("../..").resolve())
if _pkg_root not in sys.path:
    sys.path.insert(0, _pkg_root)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", _pkg_root],
               capture_output=True, text=True)

import numpy as np
import pandas as pd
from IPython.display import Image, display

import SelectOmics
from SelectOmics import SelectOmicsConfig, SelectOmicsPipeline

SelectOmics.enable_logging("INFO")     # the pipeline reports through logging
print(f"SelectOmics {SelectOmics.__version__} · Python {sys.version.split()[0]}")

## 2 · Data

The `_aligned.csv` files are stored features by samples, so they are transposed on load and the labels are attached by row order. The prepared `_input.csv` is cached after the first run.

In [ ]:
DATA_DIR = Path(".").resolve()
TARGET   = "Label"

# The MLOmics data is not committed to this repository. mlomics_data fetches it
# from a pinned, immutable dataset revision on first use, verifies each file
# against a recorded SHA-256, and caches it under examples/.mlomics_cache/.
# See examples/mlomics_data.py for the source, the licence and the citation.
if str(DATA_DIR.parent) not in sys.path:
    sys.path.insert(0, str(DATA_DIR.parent))
from mlomics_data import build_input, build_merged

CSV = DATA_DIR / "GBM_CNV_input.csv"

if not CSV.exists():
    # build_input transposes the published features-by-samples matrix, adds
    # the layer suffix, and attaches the labels by row order.
    build_input("GBM", "CNV").to_csv(CSV, index=False)
    print(f"built {CSV.name} from the pinned MLOmics revision")

df = pd.read_csv(CSV)

n_samples, n_features = df.shape[0], df.shape[1] - 1
print(f"{n_samples} samples x {n_features} features")
print(f"n/p ratio    : {n_samples / n_features:.4f}")
print(f"missing      : {100 * df.drop(columns=[TARGET]).isna().mean().mean():.3f}%")

counts = df[TARGET].value_counts().sort_index()
print("\nclass balance:")
for cls, k in counts.items():
    print(f"  class {cls}: {k:>3}  ({100 * k / n_samples:4.1f}%)")
print(f"imbalance    : {counts.max() / counts.min():.2f}:1")

## 3 · Configure

`suggest()` inspects the file and proposes settings, reporting what it saw and
why. Take it as a starting point and override what you disagree with.

In [ ]:
suggested = SelectOmicsConfig.suggest(str(CSV), TARGET)
print(f"suggested algorithm        : {suggested.algorithm}")
print(f"suggested consensus models : {suggested.n_consensus_models}")
print(f"suggested tuning iterations: {suggested.quick_tune_iterations}")
print(f"suggested bootstrap        : {suggested.n_bootstrap}")

In [ ]:
config = SelectOmicsConfig(
    data_path=str(CSV),
    target_column=TARGET,
    algorithm="XGB",
    output_dir="results_GBM_CNV",

    # Kept modest so the notebook finishes in reasonable time. Raise both for
    # published work.
    n_consensus_models=5,
    quick_tune_iterations=10,
    n_bootstrap=30,
    random_seed=42,

    enable_step1=True,
    enable_step2=True,
    enable_step3=True,

    # --- every evaluation this pipeline offers ---
    enable_step_evaluations=True,        # each step against the reference
    enable_final_test_evaluation=True,   # held-out metrics, recommended panel
    enable_nested_cv=False,   # see the note below
    

    create_visualizations=True,
    save_intermediate_results=True,
    verbose=True,
)
print(f"output directory : {config.output_dir}")
print(f"algorithm        : {config.algorithm}")
print(f"nested CV        : {config.enable_nested_cv}")

**Why nested CV is off here.** `enable_nested_cv=True` reruns the whole
procedure once per outer fold, selection, validation and recommendation alike.
Measured on the smallest layer in this cohort it took 282 minutes against 60
for the run on its own, and this layer carries 11,205 features. It is the right
tool for confirming a result you intend to publish, and
`../GS-OV/OV_SelectOmics_Showcase.ipynb` reads its output in detail.

## 4 · Run

`validate=True` runs the three validation protocols against **every** step's
panel and builds the recommendation from that comparison. Without it you get
panels but no verdict, and no held-out evaluation.

In [ ]:
t0 = time.perf_counter()
pipeline = SelectOmicsPipeline(config)
results  = pipeline.run(validate=True)
elapsed  = time.perf_counter() - t0

print(f"\ncompleted in {elapsed / 60:.1f} min")
print("result keys:", sorted(results.keys()))

## 5 · Each step against the reference

`enable_step_evaluations=True` makes every step evaluate its own panel with the
same cross-validation used for the Step 0 reference, so the table below says
whether a step helped rather than only how much it removed.

In [ ]:
ref = results["step0"]["reference_result"]["mean_auc"]
rows = []
for key in ("step0", "step1", "step2", "step3"):
    r = results.get(key)
    if r is None:
        rows.append({"step": key, "status": "not run"})
        continue
    if r.get("skipped"):
        rows.append({"step": key, "status": "skipped (below Step 3's gate)",
                     "n_features": len(pipeline.get_selected_features(key))})
        continue
    # A step that ran files its own evaluation under consensus_result; Step 0
    # files the reference it is compared against. A step that skipped is
    # handled above, because it files the reference result too and reading it
    # here would credit it with the full-feature AUC.
    ev = r.get("consensus_result") or r.get("reference_result") or {}
    auc = ev.get("mean_auc", float("nan"))
    rows.append({
        "step": key,
        "n_features": len(pipeline.get_selected_features(key)),
        "mean_auc": round(auc, 4),
        "vs_reference": round(auc - ref, 4),
        "outcome": r.get("consensus_outcome", ""),
        "agreement": r.get("agreement_label", ""),
    })
display(pd.DataFrame(rows))

## 6 · Validation, and the recommendation

Stratified CV, leave-one-out and bootstrap run against every step's panel. The
recommendation reduces that table to a verdict: the smallest panel whose score
is within one standard error of the best.

It is the panel to use. `get_selected_features()` returns the last step, which
is the same thing only when the last step also validated best.

In [ ]:
comparison = results["validation"]["comparison_df"]
cols = ["n_features", "cv_auc", "cv_std", "loo_auc", "bootstrap_auc",
        "bootstrap_ci_lower", "bootstrap_ci_upper", "ci_width_flag"]
display(comparison[[c for c in cols if c in comparison.columns]].round(4))

In [ ]:
rec = results["recommendation"]
last = pipeline.get_selected_features()
recommended = pipeline.get_recommended_features()

print(f"recommended step : {rec['step_id']}")
print(f"features         : {rec['n_features']}")
print(f"ranked on        : {rec['ranking_basis']}")
print(f"score            : {rec['ranking_score']:.4f}")
print(f"quality          : {rec['quality']}")
print(f"reason           : {rec['reason']}")

print(f"\nlast step's panel : {len(last):>5} features")
print(f"recommended panel : {len(recommended):>5} features")
print(f"identical         : {set(last) == set(recommended)}")

# The guard: a panel scoring far above the unselected Step 0 on the same
# samples that selected it is more often optimism than signal.
if rec.get("selection_optimism_suspected"):
    print(f"\nselection-optimism guard FIRED "
          f"(gain over {rec['reference_id']}: {rec['selection_gain']:.3f})")
else:
    print(f"\nselection gain over {rec['reference_id']}: "
          f"{rec['selection_gain']:.3f}  (within the plausible range)")

## 7 · Held-out test, on the recommended panel

The first and only look at data no protocol above has seen. It scores the
**recommended** panel, and the training CV AUC it is compared against comes
from that same panel, so the generalisation gap is like for like.

**Read the gap, not just the AUC.** Everything before this point was computed
on the training split, where the features were also chosen, so all of it is
optimistic to some degree. The gap is what says by how much:

| Gap | Label | What it means |
|---|---|---|
| < 0.05 | excellent | The training estimate transferred |
| 0.05 to 0.10 | good | Minor optimism, normal at this sample size |
| 0.10 to 0.15 | moderate | Treat the training numbers as an upper bound |
| > 0.15 | poor | The panel did not transfer; do not rely on it |

A poor gap is a real result rather than a failure of the run, and it is worth
seeing at least once. In this cohort `OV_SelectOmics_CNV` produces one: its
panel validated at 0.81 and scored 0.59 on held-out samples, a gap of 0.23.
The selection-optimism guard in Section 6 stays quiet there and is right to,
because that guard compares the panel against the *unselected* Step 0 panel,
and on that layer Step 0 is optimistic too. Optimism shared by every panel is
exactly what the held-out test exists to catch.

In [ ]:
def _f(v):
    return "n/a" if v is None else f"{v:.4f}"

ft = results["final_test"]
m = ft["metrics"]
print(f"panel scored       : {ft['panel']}, {ft['step_name']} "
      f"({ft['n_features']} features)")
print(f"test AUC           : {_f(ft['test_auc'])}")
print(f"balanced accuracy  : {_f(m.get('balanced_accuracy'))}")
print(f"macro F1           : {_f(m.get('macro_f1'))}")
print(f"training CV AUC    : {_f(m.get('training_cv_auc_mean'))}  (same panel)")
print(f"generalisation gap : {_f(m.get('generalization_gap'))}  ({ft['gen_label']})")
print(f"test samples       : {len(pipeline._y_test)}")

adequacy = results.get("sample_adequacy")
if adequacy:
    print(f"\nsample adequacy    : {adequacy.get('overall_severity')}")
    for w in adequacy.get("warnings", []):
        print(f"  - {w}")

## 8 · Feature provenance

Which step dropped each feature, and where the survivors came from.

In [ ]:
prov = pipeline.get_feature_provenance()
print(f"{len(prov)} features tracked")
display(prov.head(10))

survived = prov["n_steps_survived"].value_counts().sort_index()
print("\nfeatures by number of steps survived:")
for k, v in survived.items():
    print(f"  {k} steps: {v}")

## 9 · Figures and files

Every figure the run wrote, checked against what the enabled flags should have
produced. A MISSING line means a figure the configuration asked for did not
appear.

In [ ]:
out = Path(config.output_dir)
expected = [
    "pipeline_summary", "validation_comparison",
    "step0_reference_roc", "step0_reference_boxplots",
    "learning_curve_roc", "roc_confusion",
]
for key in ("step1", "step2", "step3"):
    if key in results and not results[key].get("skipped"):
        expected += [f"{key}_consensus_vs_reference_roc",
                     f"{key}_consensus_vs_reference_boxplots"]

fmt = config.plot_format
missing = [stem for stem in expected if not (out / f"{stem}.{fmt}").exists()]
for stem in expected:
    f = out / f"{stem}.{fmt}"
    status = "ok     " if f.exists() else "MISSING"
    size = f"{f.stat().st_size / 1024:8.1f} KB" if f.exists() else ""
    print(f"  {status} {stem}.{fmt:<4} {size}")
print(f"\n{len(expected) - len(missing)} of {len(expected)} figures present")

In [ ]:
# The two that summarise the run: the step trajectory and the validation
# comparison the recommendation was built from.
for stem in ("pipeline_summary", "validation_comparison"):
    f = out / f"{stem}.{config.plot_format}"
    if f.exists():
        display(Image(filename=str(f)))

In [ ]:
pipeline.save_results()
for f in sorted(out.glob("*")):
    if f.is_file():
        print(f"  {f.name:<48} {f.stat().st_size / 1024:>9.1f} KB")

In [ ]:
# The panel to carry forward, annotated with where each feature was dropped or
# kept. recommended_features.csv holds the same list on its own.
annotated = prov[prov["feature"].isin(recommended)].copy()
annotated.to_csv(out / "recommended_panel_annotated.csv", index=False)
print(f"wrote {out / 'recommended_panel_annotated.csv'} "
      f"({len(annotated)} features)")
display(annotated.head(10))

## 10 · Next

- `GBM_SelectOmics.ipynb` runs the same pipeline on all four layers at
  once, and reports which layer each recommended feature came from.
- `../GS-OV/OV_SelectOmics_Showcase.ipynb` compares the panel against LASSO,
  ElasticNet and RFECV, ablates the steps, and reads every evaluation module in
  detail.
- `benchmarks/BENCHMARKS.md` carries the measured behaviour this notebook
  relies on: what the recommendation is worth (section 8), and when to trade
  precision for recall (section 7.3).